# QAOA vs. Classical Heuristic on MaxCut - Rui Qian Khor, William Arms-Roberts



We implimented QAOA using qiskit to solve the MaxCut problem. The quantum algorithm was compared to the classical greedy Sahni-Gonzalez Algorithm.

We measured the solution quality of the results of each algorithm through the approximation ratio. To find the true optimal MaxCut, a classical brute force algorithm was utlized.


## Initialize Programs and Libraries




In [1]:
!pip3 install --upgrade pip
!pip install networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
!pip install qiskit==1.3
!pip install qiskit-aer==0.15
!pip install pylatexenc==2.10
!pip install qiskit_ibm_runtime==0.34.0

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_ibm_runtime.fake_provider import FakeSherbrooke  # choosing fake backend
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeSherbrooke()
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 46.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 12.7 MB/s  0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=db58ff2a3135d32797e8294e7b9b13d2cdf2acf56a11bae9fe74f6c5a12f431e
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.7 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installa

In [3]:
!pip freeze > requirements.txt

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the target path for saving results
target_path = '/content/drive/MyDrive/QAOA Testing'

# Create the directory if it doesn't exist
if not os.path.exists(target_path):
    os.makedirs(target_path)
    print(f"Created directory: {target_path}")
else:
    print(f"Directory already exists: {target_path}")

Mounted at /content/drive
Directory already exists: /content/drive/MyDrive/QAOA Testing


# Graph generator

In [ ]:
import networkx as nx
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph

def create_graph(nodes, edges, filename=None):
  nx_graph = nx.random_regular_graph(edges, nodes) #creates a random graph using networkx
  graph = rx.PyGraph()
  graph.add_nodes_from(range(nodes))
  for u, v in nx_graph.edges(): # converts the networkx graph to rustworkx
    graph.add_edge(u, v, 1.0)

  if filename:
    nx.write_gml(nx_graph, filename)

  return graph

# Quantum Approximation Optimization Algorithm

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,) -> list[tuple[str, list[int], float]]:
    """
    Inputs: rustworkx graph

    Builds a pauli list of gates for the circuit
    Returns the pauli-list
    """

    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator ):
    """
    Inputs: Gamma and beta circuit parameters, the ansatz circuit, hamiltonian corresponding to the graph, and the estimator primative

    Returns the estimated value of the cost hamiltonian by measuring the circuit with the inputed gamma and beta parameters

    """

    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout) # Matches the given hamiltonian to the given circuit

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub]) #Estimates the expected value of the cost hamiltonian

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals = []

    objective_func_vals.append(cost)

    return cost

In [ ]:
def optimize(shots, cost_func_estimator, init_params, candidate_circuit, cost_hamiltonian ):
  """
  Inputs: # of shots, estimator function, the inital gamma and beta parameters, the ansatz circuit whose parameters are being optimized, The cost hamiltonian corresponding to the max-cut graph

  Runs the cost_func_estimator function repeatedly with a classical optimzer to find the optimzed gamma and beta parameters to use for the final circuit

  Returns the final optimized parameters
  """
  from qiskit_ibm_runtime import SamplerV2 as Sampler
  from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
  from qiskit_aer import AerSimulator

  backend = AerSimulator(method="statevector")

  estimator = Estimator(mode=backend)
  estimator.options.default_shots = shots

  result = minimize(
    cost_func_estimator,
    init_params,
    args=(candidate_circuit, cost_hamiltonian, estimator),
    method="COBYLA",
    tol=1e-2,
  )

  return (result)

In [ ]:
def run_circuit(optimized_circuit):
  """
  Inputs: The final optimzed QAOA circuit

  Runs the circuit through the AerSimulator and records the output distributions

  Returns the final distribution of outputs in both integer and bitstring form

  """
  from qiskit_ibm_runtime import SamplerV2 as Sampler
  from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
  from qiskit_aer import AerSimulator
  backend = AerSimulator()

  sampler = Sampler(mode=backend)
  sampler.options.default_shots = 1000

  pub = (optimized_circuit,)
  job = sampler.run([pub], shots=int(1e4))
  counts_int = job.result()[0].data.meas.get_int_counts()
  counts_bin = job.result()[0].data.meas.get_counts()
  shots = sum(counts_int.values())
  final_distribution_int = {key: val / shots for key, val in counts_int.items()}
  final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
  final_distribution_bin = dict(sorted(final_distribution_bin.items(), key=lambda item: item[1], reverse=True))
  final_distribution_int = dict(sorted(final_distribution_int.items(), key=lambda item: item[1], reverse=True))

  return final_distribution_int, final_distribution_bin

In [ ]:

def to_bitstring(integer, num_bits):
  """
  Inputs: A single digit representing a measurement outcome (i.e from final_distribution_int), the number of qubits in the circuit

  Returns a list of 1's and 0's representing the graph cut, grouping the nodes into two sets

 """
  result = np.binary_repr(integer, width=num_bits)
  return [int(digit) for digit in result]

def result_bitstring(final_distribution_int, nodes):

  keys = list(final_distribution_int.keys())
  values = list(final_distribution_int.values())
  most_likely = keys[np.argmax(np.abs(values))]
  most_likely_bitstring = to_bitstring(most_likely, nodes)
  most_likely_bitstring.reverse()

  return most_likely_bitstring

In [ ]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
  """
  Inputs: a list of 0's and 1's representing the cut of the graph, The rustworkx graph that the cut will be evaluated on

  Returns the value of the cut on the graph

  """
  assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
  return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )

In [ ]:
def show_result_distribution(final_distribution_bin): # plots the final distribution, we didnt use this since the number of outputs made the graphs very cluttered
  matplotlib.rcParams.update({"font.size": 10})
  total_bits = final_distribution_bin
  values = np.abs(list(total_bits.values()))
  total_bits = dict(sorted(final_distribution_bin.items(), key=lambda item: item[1], reverse=True))
  final_bits = {k: v for i, (k, v) in enumerate(total_bits.items()) if i < 10}
  fig = plt.figure(figsize=(11, 6))
  ax = fig.add_subplot(1, 1, 1)
  plt.xticks(rotation=45)
  plt.title("Result Distribution")
  plt.xlabel("Bitstrings (reversed)")
  plt.ylabel("Probability")
  ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
  plt.show()

In [ ]:
def QAOA(graph, nodes, edges, count):
  """
  Inputs: The rustworx graph, the number of nodes in the graph, the number of nodes in the graph, the number of edges per node, The run iteration (used for numbering the raw data)

  Combines the functions defined above to run the full QAOA circuit from start to finish

  Returns the final cut value, the number of two-qubit gates, the total number of gates
  """
  from qiskit.qasm3 import dumps
  import json

  max_cut_paulis = build_max_cut_paulis(graph)
  cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, nodes)

  circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
  circuit.measure_all()

  candidate_circuit = pm.run(circuit)

  initial_gamma = np.pi
  initial_beta = np.pi / 2
  init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

  shots = 1000

  optimization_result = optimize(shots, cost_func_estimator, init_params, candidate_circuit, cost_hamiltonian)

  optimized_circuit = candidate_circuit.assign_parameters(optimization_result.x)

  final_distribution_int, final_distribution_bin = run_circuit(optimized_circuit)

  most_likely_bitstring = result_bitstring(final_distribution_int, nodes)

  cut_value = evaluate_sample(most_likely_bitstring, graph)

  gate_counts        = dict(optimized_circuit.count_ops())
  total_gate_count   = sum(gate_counts.values())
  circuit_depth      = optimized_circuit.depth()
  two_qubit_count    = gate_counts.get("cx", 0) + gate_counts.get("ecr", 0) + gate_counts.get("cz", 0)
  print(f'depth: {circuit_depth}')

  # Save optimized circuit drawing
  circuit_image_filename = f"{target_path}/image_circuit_{nodes}_{edges}_{count}.png"
  optimized_circuit.draw("mpl", fold=False, idle_wires=False, filename=circuit_image_filename)

  # Save optimized circuit as QASM
  circuit_qasm_filename = f"{target_path}/qasm_circuit_{nodes}_{edges}_{count}.qasm"
  with open(circuit_qasm_filename, "w") as f:
      f.write(dumps(optimized_circuit)) # Changed to dumps(optimized_circuit)

  results_filename = f"{target_path}/results_{nodes}_{edges}_{count}.json"

  # Collect all data
  result_data = {
      "graph_filename": "graph_{nodes}_{edges}_{count}.gml",
      "results_filename": "results_{nodes}_{edges}_{count}.json",
      "nodes": nodes,
      "graph_degree": edges, # Renamed for clarity to reflect 'k' in random_regular_graph
      "graph_connections": [list(edge) for edge in graph.edge_list()], # Added actual graph connections
      "optimized_parameters": optimization_result.x.tolist(),
      "final_cost": float(optimization_result.fun),
      "most_likely_bitstring": most_likely_bitstring,
      "cut_value": float(cut_value),
      "final_distribution_bin": final_distribution_bin,
      "image_optimized_circuit": circuit_image_filename, # New: path to circuit image
      "qasm_optimized_circuit": circuit_qasm_filename # New: path to circuit QASM
  }

  with open(results_filename, 'w') as f:
      json.dump(result_data, f, indent=4)
  print(f"Circuit results saved to {results_filename}")

  return cut_value, two_qubit_count, total_gate_count

# Classical Sahni-Gonzalez Algorithm

In [ ]:
def greedy_classical(graph):
  """
  Inputs: A rustworkx graph

  Returns an approximation of the maximum cut value
  """
  def dict_gen (graph): #converts the edges to a dictionary to use in the rest of the funciton
    G = {}
    for i, j, w in graph.weighted_edge_list():
      G[(i,j)] = 1.0
    return G

  G = dict_gen(graph)
  vertices = set()
  for (u,v) in G: #creates a set of all the vertices
    vertices.add(u)
    vertices.add(v)
  def set_weights(s1,s2): # Evaluates and returns the cut value of two sets
    total = 0.0
    for (u,v), w in G.items():
      if (u in s1 and v in s2) or (u in s2 and v in s1):
        total += w
    return total
  def w_to_set(i,s): #Evaluates the total connections a set of two vertices has to the rest of the graph
    total = 0.0
    for j in s:
      if (i,j) in G:
        total += G[(i,j)]
      elif (j,i) in G:
        total += G[(j,i)]
    return total
  max_edge = list(G.keys())[0] #Initializes the maximum edge as the first pair (This is done because we are using unweighted graphs)
  i1 = max_edge[0]
  i2 = max_edge[1]
  w = 1.0
  v1 = {i1}
  v2 = {i2}
  cut_w = 1.0
  v_prime = vertices - {i1,i2}

  for k in range(len(vertices) - 2): #Searchs through the vertices that are not the first pair, checking their weights to each of the first pair and then assigning the vertex to one of the two sets based on which weight value is higher
    if not v_prime:
      break
    scores = {}
    for i in v_prime:
      wi_v1 = w_to_set(i,v1)
      wi_v2 = w_to_set(i,v2)
      scores[i] = max(wi_v1,wi_v2)
    i_star = max(scores, key = lambda i: scores[i])
    wi_star_v1 = w_to_set(i_star,v1)
    wi_star_v2 = w_to_set(i_star,v2)
    if wi_star_v1 >= wi_star_v2:
      v2.add(i_star)
      cut_w += wi_star_v1
    else:
      v1.add(i_star)
      cut_w += wi_star_v2
    v_prime.remove(i_star) # Removes the vertex from the list
  return cut_w

# Classical Brute-Force Algorithm

In [ ]:
from itertools import product

def brute_force(graph):
  """
  Inputs: A rustworkx graph

  Returns the optimal cut value of the given graph
  """
  n = graph.num_nodes()
  best_cut = 0
  best_partition = None

  for bits in product([0, 1], repeat=n):# generates all the possible combinations of 0 and 1 for n bits
      cut_value = 0
      for u, v, w in graph.weighted_edge_list(): #iterates over each bit in the string and adds the cut value
          if bits[u] != bits[v]:
              cut_value += w
      if cut_value > best_cut: # Updates the max-cut value
          best_cut = cut_value
          best_partition = bits

  set_0 = [i for i, b in enumerate(best_partition) if b == 0] # divides all the nodes in the best parition into two sets
  set_1 = [i for i, b in enumerate(best_partition) if b == 1]

  print(f"Best cut value:  {best_cut}")

  return best_cut

# Testing

In [ ]:
#Runs a single MaxCut graph
#loop this in a main function
def test(nodes, edges, count):
  """
  Inputs: the number of nodes in the graph, the number of edges in the graph, the run iteration (used for counting the runs for the raw data)

  Runs the classical brute force, classical approximation, and QAOA and calculates the approximation ratio's

  Returns the QAOA's approximation ratio, the SG1's approximation ratio, the true optimal cut value, the number of two-qubit gates in the circuit, and the total number of gates in the quantum circuit
  """

  graph_filename = f"{target_path}/graph_{nodes}_{edges}_{count}.gml"
  graph = create_graph(nodes, edges, filename=graph_filename)
  #Runs the classical brute force algorithm to find the optimal cut value
  true_optimal = brute_force(graph)

  #Runs to find QAOA's solution
  QAOA_best,twogatez,totalgatez = QAOA(graph, nodes, edges, count)
  QAOA_AR = QAOA_best/true_optimal

  #Runs to find classical SG Algorithm's solution
  classical_best = greedy_classical(graph)
  classical_AR = classical_best/true_optimal

  return QAOA_AR, classical_AR, true_optimal,twogatez, totalgatez

In [ ]:


def main(nodes,edges, runs=1):
  """
  Inputs: the nubmer of nodes in the graph, the number of edges per node, the number of runs you want

  Runs the test function repeatedly and records the values from each run

  Returns a jsn file that contains the graph details, the optimal cut value, and the average approximation ratio for the classical agorithm and the QAOA
  """
  import gc
  import json
  from qiskit.qasm3 import dumps
  QAR = []
  CAR = []
  optimal = []
  twogates = 0
  totalgate = 0
  for i in range(runs): # Runs the QAOA and classical algorithm
    QAOA_AR,classical_AR,true_optimal,two_gate, total_gates = test(nodes,edges,i)
    print("QAOA approximation ratio:", QAOA_AR, "classical approximation ratio:", classical_AR)
    QAR.append(QAOA_AR)
    CAR.append(classical_AR)
    optimal.append(true_optimal)
    twogates += two_gate
    totalgate += total_gates

  QAR_av = (sum(QAR)/len(QAR)) # Calculates the average approximation ratio's and the average number of two-qubit gates and total number of gates
  CAR_av = (sum(CAR)/len(CAR))
  twoavg = twogates/runs
  totavg = totalgate/runs
  print(f'two gate: {twoavg}')
  print(f'total gates: {totavg}')
  average = [QAR_av, CAR_av]

  result_data = {
      "Graph Instance": f"Nodes: {nodes} Edges: {edges}",
      "Optimal Cut": optimal,
      "Average Approximation Ratio": f"QAOA: {QAR_av}, Classical: {CAR_av}",
      "QAOA Approximation Raito": QAR,
      "Classical Approximation Raito": CAR,
  }

  output_filename = f'{target_path}/QAR_axis_results_{nodes}_{edges}.json'
  with open(output_filename, 'w') as f:
      json.dump(result_data, f, indent=4)
  print(f"QAR_axis saved to {output_filename}")
  return result_data

In [ ]:
main(6,3)

Best cut value:  7.0
depth: 277
Circuit results saved to /content/drive/MyDrive/QAOA Testing/results_6_3_0.json
QAOA approximation ratio: 1.0 classical approximation ratio: 1.0
two gate: 67.0
total gates: 512.0
QAR_axis saved to /content/drive/MyDrive/QAOA Testing/QAR_axis_results_6_3.json


{'Graph Instance': 'Nodes: 6 Edges: 3',
 'Optimal Cut': [7.0],
 'Average Approximation Ratio': 'QAOA: 1.0, Classical: 1.0',
 'QAOA Approximation Raito': [1.0],
 'Classical Approximation Raito': [1.0]}